In [2]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import networkx as nx
import random
import heapq
import collections

### All the simulation uses hours and kilometers

In [3]:
LAMBDA_T = [314.2, 162.4, 138.6, 148.8, 273.2, 1118.8, 2773.8, 4036.2, 4237.4, 3277.0, 2843.0, 2876.4, 3143.0, 3277.8, 3546.2, 4335.0, 4945.4, 4525.8, 2847.8, 1828.0, 1378.4, 1271.2, 1171.2, 767.6 ]

In [4]:
#Sampling arrival times of cars to network
def lambdat(t : np.array):
    lambdat = []
    for time in t:
        lambdat.append(LAMBDA_T[int(np.floor(time))])
    return lambdat

def arrival_times(lam): #Taken from lecture notes
    max_T = 24
    arrival_times = collections.deque()
    exp_dist = stats.expon(scale = 1/lam)
    t = exp_dist.rvs()
    while t < max_T:
        arrival_times.append(t)
        t += exp_dist.rvs()
    
    return np.asarray(arrival_times)

In [5]:
Graph = nx.read_gml('networkAssignment.gml')
JUNCTIONS = list(Graph.nodes)

In [6]:
Graph.edges[('1410566272', '8432860337')]

{'name': 'Knooppunt Gouwe->Knooppunt Terbregseplein',
 'highway': 'motorway_link',
 'length': 11557.0,
 'lanes': 2}

In [7]:
class FES:
    def __init__(self):
        self.events = []

    def add(self, event):
        heapq.heappush(self.events, event)
    
    def next(self):
        return heapq.heappop(self.events)
    
    def isEmpty(self):
        return len(self.events) == 0
    
    def __repr__(self):
        string = ''
        sorted_events = sorted(self.events)
        for event in sorted_events:
            string += f'{event}\n'
        return string

In [8]:
class Event:
    TYPE = ['New car', 'Car departure', 'Accident']
    def __init__(self, typ:int, time, car = None, road = None):
        #types:
            #0 : Arrival of car to the network
            #1 : Car leaves current road and goes on to the next
            #2 : Accident in road
        self.type = typ
        self.time = time
        self.road = road

        if typ == 0:
            car = Car(time_entrance = time)
    
        self.car = car
        
    def __str__(self):
        if self.type == 0:
            return f'{self.TYPE[self.type]} from {self.car.origin} to {self.car.destination} at {self.time}'
        if self.type == 1:
            return f'{self.TYPE[self.type]} of {self.car} at {self.time}h'
        if self.type == 2:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'

    def __lt__(self, other):
        return self.time < other.time

In [9]:
class Car:
    VELOCITIES = [100, 80]
    VELOCITIES_P = [0.9, 0.1]
    def __init__(self, time_entrance, origin = None, destination = None):
        #Origin and destination
        origin, destination = np.random.choice(JUNCTIONS, 2, replace = False)
        
        self.origin = origin
        self.destination = destination

        #path to follow
        self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')

        #Velocity
        self.velocity = np.random.choice(self.VELOCITIES, p=self.VELOCITIES_P)

        #Variable to keep track how far into the path we are (to simplify scheduling events)
        #Int between 0 and len(path) - 1 that indicates in which edge we are, starting at 0
        #Essentially, how many edges has it travelled so far
        self.progress = 0
        
        #Time entrance
        self.time = time_entrance

        #Schedule next event and store it as attribute
        self.next_event = self.schedule_event_exit()


    def __str__(self):
        return f'Vehicle travelling from {self.origin} to {self.destination} at {self.velocity} km/h, atm at {self.path[self.progress]}'
    
    def schedule_event_exit(self):
        if  self.progress < len(self.path) - 1: 
            #find next edge to travel through and its length
            edge = Graph.edges[(self.path[self.progress], self.path[self.progress + 1])]
            length = edge['length']

            #Sample travel time of edge
            mean = length / (self.velocity /3.6) #seconds
            std = mean / 20
            time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours

            new_time = self.time + time_to_travel

            #Store event and increase progress
            self.next_event = Event(1 , new_time, car=self)
            self.increase_progress()
            self.increase_time(new_time)

            return self.next_event
        
        # if self.progress == len(self.path) - 1:
        #     print('Car has reached its destination')
        #     self.travel_time = self.next_event.time
    
    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time

In [10]:
#Using a thining approach
max_lambda = np.max(LAMBDA_T) + 1
all_arrivals = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_arrivals))
accept_filter = u_rvs * max_lambda < lambdat(all_arrivals)

accepted_arrivals = all_arrivals[accept_filter]

In [11]:
#Simulation (can be turned into an object later)
LIST_CARS = []
fes = FES()
for arrival in accepted_arrivals:
    #Two events associated with each arrival
    arrival_event = Event(0, arrival)
    fes.add(arrival_event)

In [12]:
t = 0 #current time
while t < 24.0:
    event = fes.next()
    t = event.time

    if event.type == 0:
        car_travel_event = event.car.next_event
        fes.add(car_travel_event)
        LIST_CARS.append(event.car)

    if event.type == 1:
        next_travel_event_car = event.car.schedule_event_exit()
        if type(next_travel_event_car) == Event: #If the car has arrived to its destination it wont return an event object
            fes.add(next_travel_event_car)
        # else:
            # print(event.car)

    # if event.type == 2: NO ACCIDENTS YET

In [14]:
#Checking if cars make it to destionation
for i in range(0, len(LIST_CARS)):
    car_i = LIST_CARS[i]
    # if car_i.progress != len(car_i.path) - 1:
    #     print(f'oh oh {LIST_CARS[i].time}')
    if car_i.path[car_i.progress] != car_i.destination:
        print(f'oh oh {LIST_CARS[i].time}')

oh oh 24.049272363376993
oh oh 24.016866939037765
oh oh 24.04986928509827
oh oh 24.077792638671088
oh oh 24.158603260397694
oh oh 24.069815540560295
oh oh 24.113450898498442
oh oh 24.05630920779861
oh oh 24.00164271912471
oh oh 24.01964969499427
oh oh 24.010244519621807
oh oh 24.035828461074857
oh oh 24.087040651957306
oh oh 24.03912801034639
oh oh 24.080223956561568
oh oh 24.022067979294587
oh oh 24.007805091252965
oh oh 24.05071576463197
oh oh 24.007074170845307
oh oh 24.05949035736109
oh oh 24.03383226509068
oh oh 24.01767967048766
oh oh 24.071036051640846
oh oh 24.127394826058953
oh oh 24.069267484214958
oh oh 24.083202466169478
oh oh 24.13464312572641
oh oh 24.00877190332172
oh oh 24.00336805807398
oh oh 24.029618018838082
oh oh 24.026682090854234
oh oh 24.098114118401924
oh oh 24.113587576047482
oh oh 24.13704768466489
oh oh 24.047821870400693
oh oh 24.132239128055257
oh oh 24.047109703170914
oh oh 24.021381573050423
oh oh 24.008060203707668
oh oh 24.24111959349876
oh oh 24.05049